In [37]:
# st_count = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_spatial_count.txt", sep="\t")
# print(st_count.shape)
# st_location = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_Locations.txt", sep="\t")
# st_density = pd.read_csv("/mnt/d/ST_Graduation_Project_data/STARmap/combined_spot_clusters.txt", sep="\t", index_col=0)
# adata = sc.AnnData(X=st_count)
# adata.uns['density'] = st_density
# adata.obsm['spatial'] = st_location.values
# print(adata)
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad")

In [38]:
# import scanpy as sc
# import pandas as pd
# adata =sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad")
# adata.obs['cell_type']=adata.obs['celltype']
# print(adata.obs['cell_type'].value_counts())
# adata.write_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad")
# adata_st = sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad")
# st_label = adata_st.uns['density']
# st_label_df = pd.DataFrame(st_label)
# st_label_df.to_csv("/mnt/d/ST_Graduation_Project/SC_MAP_ST/evaluate/STARmap_density.csv")

In [ ]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad" \
    --n_epochs 150 \
    --resolution 4 \
    --top_n_per_type 100 \
    --loss_type zinb \
    --latent_dim 256 \
    --output_dir ../deconv_results/STARmap
#    --marker_selection_method l1 
#    --precomputed_marker_file "../deconv_results/DATA18_backup/final_genes.txt"

Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 100
   Leiden resolution: 4.0
   Batch size: 256
   Epochs: 150
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: MSE
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/STARmap/scRNA.h5ad
   SC shape: (14249, 34041)
   SC avg counts/cell: 1594647.9 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad
   ST shape: (189, 882)
   Common genes: 882
   SC final: (14249, 882)
   ST final: (189, 882)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 86 clusters
Marker genes per cluster:
   0: 100 -> 100 (after correlation)
   1: 100 -> 100 (after correlation)
   10: 100 -> 100 (after correlation)
   11: 100 -> 100 (after correlation)
   12: 100 -> 100 (after correlation)
   13: 1

VAE Training: 100%|██████████| 150/150 [02:45<00:00,  1.10s/epoch, Train=169408.4704, Recon=169304.4968, KL=1039.7285, MMD=0.1673, Test=102744.1724] 


📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (14249, 775)
   Number of clusters: 86
   Computed embeddings shape: (14249, 256)
Computing cluster centers with mean aggregation...
   Completed: 86 clusters with mean centers and expressions
Extracting celltype-cluster mapping (using 'cell_type' column)...
Visualizing modality alignment...
Generating UMAP visualization for modality alignment...
   Computing UMAP on 14293 samples with 256 dims...
   UMAP visualization saved to: ../deconv_results/STARmap/modality_alignment_umap.png
Saving model to: ../deconv_results/STARmap/final_vae.pth
   Average cell counts: 1594647.9 (for Stage 2 scale factor)
✅ Saved VAE model (weights only): ../deconv_results/STARmap/final_vae.pth
Saving cluster data to: ../deconv_results/STARmap/final_vae_cluster_data.npz
   ✓ Cluster IDs: 86
   ✓ Prototypes: (86, 256)
   ✓ Expressions (marker): (86, 775)
   ✓ Expressions (full): 86 clusters × 882 genes
   ✓ Celltype mapping: 86 clus

In [40]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/STARmap/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/STARmap/final_vae.pth" \
    --output_dir "../deconv_results/STARmap/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --loss_lambda_mse 0.0001 \
    --lr 5e-3 \
    --k_spatial 10 \
    --k_celltype 15 \
    --batch_size 512 \
    --n_epoch 300

Sample name: Spatial
Stage 1 model: ../deconv_results/STARmap/final_vae.pth
Output directory: ../deconv_results/STARmap/
Device: cpu
Weight threshold: 0.0001
Loading pretrained VAE Encoder...
   VAE architecture: 775 -> 256
   Output type: mse
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 14249 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/STARmap/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([86, 256])
   Cluster expressions (marker): torch.Size([86, 775])
   Cluster expressions (all genes): 86 × 882
   Loaded celltype mapping: 86 clusters
   Average cell counts: 1594647.9
Loaded all genes list: 882 genes
VAE Encoder loaded: 775 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '

GAT Training:  29%|██▊       | 86/300 [00:41<01:43,  2.07epoch/s, Total=3.0655, MSE=2870.4360, Spot_Cosine=0.2758, Gene_Cosine=0.2851, Pearson=0.7255, Gene_Pearson=0.8549, P_pat=8, M_pat=76, C_pat=76] 


KeyboardInterrupt: 